# Prep modelling

In [ ]:
import pandas as pd
from pipeline.feature_engineering import CustomerFeatureEngineer
from data.ingestion_pipeline import get_mysql_engine

# 1. Pull data blindly from MySQL and generate the behavioral features!
engineer = CustomerFeatureEngineer()
behavioral_features = engineer.run_feature_engineering()

# 2. Pull the base customer table (we still need the 'churn' label and age)
engine = get_mysql_engine()
customer_df = pd.read_sql_query("""
    SELECT customer_id, age, gender, city, segment_initial, tenure_months, churn 
    FROM customer_data
""", engine)

# 3. Merge them together
df_survival = pd.merge(customer_df, behavioral_features, on='customer_id', how='inner')

# 4. Clean up the non-numeric data for the CoxPH model
columns_to_drop = [
    'customer_id', 'tenant_id', 'snapshot_date', 'feature_version', 
    'first_tx', 'last_tx', 'promotion_types'
]
df_survival = df_survival.drop(columns=[col for col in columns_to_drop if col in df_survival.columns])

# 5. One-Hot Encode
df_survival = pd.get_dummies(df_survival, columns=['gender', 'city', 'segment_initial'], drop_first=True)

# You are now ready to train the CoxPH model! 
